# Preprocesamiento - notebook
El preprocesamiento es una de las fases más críticas en cualquier proyecto de Machine Learning. Los modelos aprenden patrones a partir de los datos, por lo que la calidad de los datos de entrada determina directamente la calidad de las predicciones.

In [1]:
import pandas as pd
import sklearn

Es importante que todo el preprocesamiento se diseñe exclusivamente sobre el conjunto de **train**, para luego replicar las mismas transformaciones sobre test sin introducir data leakage.

In [2]:
df_train = pd.read_csv("data/df_train_small.csv")

## 1. Selección Inicial de Variables

Usamos un fichero Excel donde previamente se ha clasificado cada variable como posible predictora o no, junto con el motivo de exclusión.

In [3]:
raw_predictors_vars = pd.read_excel("data/variables_withExperts.xlsx")
raw_predictors_vars.head()

,variable,categoria,descripcion,posible_predictora,motivo_exclusion
0,id,identificador,Identificador unico del prestamo,no,Identificador sin valor predictivo
1,member_id,identificador,Identificador unico del miembro,no,Identificador sin valor predictivo
2,loan_amnt,solicitud,Importe del prestamo solicitado,si,NaN
3,funded_amnt,post_concesion,Importe total financiado,no,Data leakage: conocido post-aprobacion
4,funded_amnt_inv,post_concesion,Importe financiado por inversores,no,Data leakage: conocido post-aprobacion


In [4]:
raw_predictors_vars = ( raw_predictors_vars
                       .query("posible_predictora == 'si'")
                        .variable
                        .tolist())
raw_predictors_vars

['loan_amnt',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'purpose',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'application_type',
 'annual_inc_joint',
 'dti_joint',
 'verification_status_joint',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'open_acc_6m',
 'open_act_il',
 'open_il_12m',
 'open_il_24m',
 'mths_since_rcnt_il',
 'total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m',
 'max_bal_bc',
 'all_util',
 'total_rev_hi_lim',
 'inq_fi',
 'total_cu_tl',
 'inq_last_12m',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_

Una vez identificadas las variables predictoras, construimos el dataset de trabajo seleccionando solo esas columnas más la variable objetivo (loan_status).

In [5]:
target_var = "loan_status"

df_train_filter_1 = df_train[ raw_predictors_vars ]
df_train_filter_1.head() 

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog
0,8000.0,36 months,7.59,249.19,A,A3,Supervisory Personal Property,10+ years,MORTGAGE,130000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,18000.0,36 months,11.53,593.83,B,B5,Terminal Manager,7 years,MORTGAGE,106340.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6200.0,36 months,10.99,202.96,B,B3,Journeyman Meatcutter,6 years,RENT,32000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10200.0,36 months,7.89,319.12,A,A5,owner,10+ years,OWN,24000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,15000.0,36 months,11.49,494.57,B,B5,Owner,10+ years,MORTGAGE,75000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df_train_filter_1.shape

(80000, 101)

Me salen 101 y antes salían 93 porque he usado withExperts

In [7]:
(df_train_filter_1.isnull().sum()/80000).sort_values(ascending=False).to_dict()

{'sec_app_mths_since_last_major_derog': 0.99705,
 'sec_app_revol_util': 0.99165,
 'sec_app_collections_12_mths_ex_med': 0.9915,
 'sec_app_chargeoff_within_12_mths': 0.9915,
 'sec_app_num_rev_accts': 0.9915,
 'sec_app_open_act_il': 0.9915,
 'sec_app_open_acc': 0.9915,
 'sec_app_mort_acc': 0.9915,
 'sec_app_inq_last_6mths': 0.9915,
 'sec_app_earliest_cr_line': 0.9915,
 'sec_app_fico_range_high': 0.9915,
 'sec_app_fico_range_low': 0.9915,
 'revol_bal_joint': 0.9915,
 'dti_joint': 0.9859625,
 'annual_inc_joint': 0.98595,
 'verification_status_joint': 0.98595,
 'mths_since_last_record': 0.82845,
 'mths_since_recent_bc_dlq': 0.760775,
 'mths_since_last_major_derog': 0.7346375,
 'il_util': 0.676475,
 'mths_since_recent_revol_delinq': 0.66205,
 'mths_since_rcnt_il': 0.63675,
 'open_acc_6m': 0.6272625,
 'open_il_24m': 0.6272625,
 'open_act_il': 0.6272625,
 'open_il_12m': 0.6272625,
 'total_bal_il': 0.6272625,
 'inq_fi': 0.6272625,
 'open_rv_12m': 0.6272625,
 'open_rv_24m': 0.6272625,
 'max_bal_

In [8]:
nulls_vars = (df_train_filter_1.isnull().sum()/80000).sort_values(ascending=False).to_frame(name="nulls_perc").reset_index()

In [9]:
nulls_vars.head()

,index,nulls_perc
0,sec_app_mths_since_last_major_derog,0.99705
1,sec_app_revol_util,0.99165
2,sec_app_collections_12_mths_ex_med,0.99150
3,sec_app_chargeoff_within_12_mths,0.99150
4,sec_app_num_rev_accts,0.99150


## Eliminación de variables con demasiados nulos

In [10]:
# Las variables con más del 99%
var_with_most_nulls = nulls_vars.query("nulls_perc >= 0.99")["index"].tolist()
df_train_filter_2 = df_train_filter_1.drop(columns=var_with_most_nulls)
df_train_filter_2.shape

(80000, 88)

## Imputación de Valores Nulos